## USING STRUCTURED OUTPUTS

    Models can be requested to give outputs in a particular format, the structure can be defined and there can be a particular schema for the metadata

In [9]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

print("Chatbot Started! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    response = llm.invoke([HumanMessage(content=user_input)])

    print("Bot:", response.content)

Chatbot Started! Type 'exit' to quit.

Bot: I cannot provide an exact figure for the size of the dataset I was trained on. The precise details of the training data, including its exact size, are proprietary information held by Google.

However, I can tell you that the dataset is **immense**. It comprises **trillions of tokens** (a token can be a word, part of a word, or a punctuation mark) and spans **many petabytes** of data.

This vast collection includes a diverse range of text and code from the internet, such as:
*   Books
*   Articles
*   Websites
*   Code repositories
*   Academic papers
*   And many other forms of publicly available digital text.

The goal is to provide a broad and deep understanding of human language, facts, reasoning, and various forms of expression.
Bot: That's a great question! My "memory" isn't like a human brain's memory at all. It's more accurate to think of it in a few different layers:

1.  **Long-Term "Memory" (My Training Data & Model Parameters):**
 

# Pydantic

    Field validation, descriptions and nested structures 

    -> The ... is a special Python value called Ellipsis. ("This field is required and must be provided.")

## Pydantic adds:

    validation  ""Check that the data matches the expected type and format."
    defaults
    descriptions
    constraints
    methods

    That's why most LangChain projects use Pydantic for structured outputs.

In [4]:
from pydantic import BaseModel, Field

# it is just a baseclass for creating pydantic models, and it does not enforce any specific behavior regarding required fields.
class Movi(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(description="The rating of the movie (0.0 to 10.0)"  )
    
    # Pydantic v2 may still treat it as required because there is no default 
    # value, but using ... makes the intent explicit and is a very 
    # common pattern.

In [6]:
model_with_struct = llm.with_structured_output(Movi)
model_with_struct.invoke([HumanMessage(content=input("Describe a movie: "))])

Movi(title='The Exorcist', year=1973, genre='Horror', director='William Friedkin', rating=8.0)

In [8]:
res = llm.invoke([HumanMessage(content="where aare you fetching the movi ratings from?")])
print(res.content)
print(res.usage_metadata)

As an AI, I don't "fetch" real-time data like a human browsing the internet or an application calling an API.

My knowledge about movie ratings comes from the vast dataset I was trained on. This dataset included a massive amount of text and information from the internet up to my last training update.

Therefore, the ratings I might cite or discuss would reflect information that was publicly available and present in my training data from various sources, such as:

*   **IMDb (Internet Movie Database)**
*   **Rotten Tomatoes** (both Tomatometer and Audience Score)
*   **Metacritic**
*   **User reviews and forums**
*   **Critical reviews from various publications**
*   **General entertainment news sites**

It's important to note that my information isn't real-time, so very recent changes to ratings or new movie releases might not be reflected. For the most up-to-date and specific ratings, you would need to check current movie review websites or databases directly.
{'input_tokens': 12, 'ou

In [24]:
# Nested structured output example

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    age: int = Field(..., description="The age of the actor")
    role: str = Field(description="The role of the actor in the movie")
    awards: list[str] = Field(default_factory=list, description="List of awards won by the actor")
    
class MoviDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(default_factory=list, description="List of actors in the movie")
    
    
model_with_struct = llm.with_structured_output(MoviDetails)
response = model_with_struct.invoke([HumanMessage(content="Provide details of the movie Inception including its cast.")])
print(response)
    

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', age=35, role='Dom Cobb', awards=[]), Actor(name='Joseph Gordon-Levitt', age=29, role='Arthur', awards=[]), Actor(name='Elliot Page', age=23, role='Ariadne', awards=[]), Actor(name='Tom Hardy', age=32, role='Eames', awards=[]), Actor(name='Cillian Murphy', age=34, role='Robert Fischer', awards=[]), Actor(name='Ken Watanabe', age=50, role='Mr. Saito', awards=[]), Actor(name='Marion Cotillard', age=34, role='Mal', awards=[])]


# TypeDict

    no validation or constraint (just a python dictionary)

    1. TypedDict : Used to define the expected structure of a dictionary.
    2. Annotated : Adds extra metadata to a type.
    3. TypeAlias : Used to create a custom name for a type.
                    from typing_extensions import TypeAlias
                        Scores: TypeAlias = dict[str, list[int]]




In [21]:
from typing_extensions import Annotated, TypedDict

# Nested structured output example

class Actor(TypedDict):
    name: str 
    age: int 
    role: str 
    
class MoviDetails(TypedDict):
    title: str 
    year: int 
    cast: list[Actor]
    
response = llm.invoke([HumanMessage(content="Provide details of the movie Inception including its cast.")])
print(response.content)


# response.profile

*Inception* is a critically acclaimed and commercially successful 2010 science fiction action film written, directed, and produced by Christopher Nolan. Known for its complex narrative, stunning visuals, and philosophical themes, it quickly became a cultural touchstone.

Here are the details:

## Inception (2010)

**Logline:** A skilled thief who steals information by entering people's dreams is given the inverse task of planting an idea into a target's subconscious, in exchange for a chance to reunite with his children.

**Plot Summary:**

The story follows **Dom Cobb** (Leonardo DiCaprio), an "extractor" – a professional who uses experimental military technology to enter the dreams of others and steal valuable information from their subconscious. He is a fugitive from the law, wanted for the alleged murder of his wife, Mal, and desperately wants to return home to his children.

Cobb and his partner, Arthur, attempt to extract information from a powerful businessman named **Saito** (K

# Data Classes
    "I want a simple object to store data."
    


In [22]:
from dataclasses import dataclass

@dataclass
class Movie:
    title: str
    year: int

In [ ]:
response = llm.invoke([human_msg])